# Clase 037 — subplots y gridspec

**Parte 0** · VanderPlas cap. 4 § 4.6.

> 🎯 Múltiples plots en una figura. Grillas regulares + GridSpec para irregulares.

> ⏱️ ~60 min

## ⚙️ Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
rng = np.random.default_rng(42)

N = 200
df = pd.DataFrame({
    'x'   : rng.normal(0, 1, N),
    'y'   : rng.normal(0, 1, N),
    'mass': rng.uniform(3000, 5000, N),
    'bill': rng.uniform(35, 50, N),
})

## 1️⃣ Grilla regular — `plt.subplots(n, m)`

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7), constrained_layout=True)
feats = ['x', 'y', 'mass', 'bill']

for ax, col in zip(axes.flat, feats):
    ax.hist(df[col], bins='auto', edgecolor='white')
    ax.set_title(col)
    ax.grid(alpha=0.3)

fig.suptitle('Distribuciones (grilla 2×2)', fontsize=14)
plt.show()

## 2️⃣ `sharex` / `sharey` — comparar con misma escala

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 4), sharey=True, constrained_layout=True)
for ax, sp in zip(axes, ['Adelie', 'Chinstrap', 'Gentoo']):
    data = rng.normal({'Adelie': 3700, 'Chinstrap': 3700, 'Gentoo': 5050}[sp], 400, 100)
    ax.boxplot(data, vert=True)
    ax.set_title(sp)
    ax.grid(axis='y', alpha=0.3)
axes[0].set_ylabel('body_mass (g)')
fig.suptitle('body_mass por species — sharey=True')
plt.show()

## 3️⃣ `GridSpec` — layouts irregulares

Scatter central + marginales arriba/derecha ("joint plot"):

In [ ]:
fig = plt.figure(figsize=(8, 8), constrained_layout=True)
gs = GridSpec(4, 4, figure=fig)

ax_main  = fig.add_subplot(gs[1:, :-1])        # filas 1-3, cols 0-2
ax_xhist = fig.add_subplot(gs[0,  :-1])         # fila 0, cols 0-2
ax_yhist = fig.add_subplot(gs[1:, -1])          # filas 1-3, col 3

# Scatter principal
ax_main.scatter(df['x'], df['y'], alpha=0.5)
ax_main.set_xlabel('x')
ax_main.set_ylabel('y')
ax_main.grid(alpha=0.3)

# Marginal X (arriba)
ax_xhist.hist(df['x'], bins='auto', color='steelblue', alpha=0.7)
ax_xhist.axis('off')

# Marginal Y (derecha) — horizontal
ax_yhist.hist(df['y'], bins='auto', color='steelblue', alpha=0.7, orientation='horizontal')
ax_yhist.axis('off')

fig.suptitle('Joint plot con GridSpec')
plt.show()

## 4️⃣ `constrained_layout` vs `tight_layout`

Ambos evitan superposición de labels/leyendas. **`constrained_layout=True`** (al crear la figura) es más nuevo y más confiable; `tight_layout()` se llama después y a veces falla con leyendas externas o colorbars.

## ✅ Checklist

- [ ] Creo grillas con `plt.subplots(n, m)`
- [ ] Itero con `axes.flat` para llenar en loop
- [ ] Uso `sharex/sharey` para comparar
- [ ] Sé construir layouts irregulares con GridSpec
- [ ] Prefiero `constrained_layout=True` a `tight_layout()`

## 📝 Homework

Ver `README.md`. Grilla 2×2 hists, 3 boxplots sharey, joint plot con GridSpec, comparativa layouts.

## 📖 Definiciones y características

**`plt.subplots(n, m)`**

Crea figure + grilla regular de n filas × m columnas de axes. Devuelve `(fig, axes)` donde axes es array 2D — itera con `.flat` para llenar.

**`sharex` / `sharey`**

Comparten escala entre axes del grid. Ideal para comparar distribuciones de la misma magnitud sin distorsión visual.

**`GridSpec`**

Layout irregular avanzado: define grilla y asigna spans manualmente ("plot grande arriba + 3 pequeños abajo"). Mucho más flexible que `subplots(n, m)`.

**`add_subplot(spec)`**

Añade un axes a una figure según una posición específica (índice 121, o GridSpec). Útil cuando subplots no alcanza.

**`constrained_layout=True`**

Algoritmo moderno (default en pandas 3+) que ajusta spacing automáticamente. Maneja colorbars, leyendas externas, suptitle sin overlap. Recomendado sobre `tight_layout()`.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| `axes` es array 1D cuando esperaba 2D | Si pides `subplots(1, 3)` o `subplots(3, 1)`, axes es 1D. **Fix**: usa `axes.flat` (siempre 1D) para iterar consistente, o `axes.reshape(-1)`. |
| `subplots(1, 1)` devuelve axes escalar | No es array. **Fix**: si quieres array siempre, `subplots(1, 1, squeeze=False)` (devuelve 2D). O distingue caso 1 vs N. |
| Plots se superponen / labels cortados | Sin `constrained_layout=True` ni `tight_layout()`. **Fix**: `subplots(..., constrained_layout=True)` siempre. |
| `sharey=True` pero un plot tiene escala distinta | Quizás esa subplot necesita un eje secundario: `ax.twinx()`. Mezclar escalas con shared rompe la utilidad. |
| GridSpec con índices confusos | El orden es `gs[fila, col]` igual que NumPy. Spans: `gs[0:2, 1]` = filas 0-1 (excluye 2), columna 1. |

## ❓ Preguntas frecuentes

**❓ ¿Cuándo `subplots(n, m)` vs `GridSpec`?**

**Regular**: `subplots`. **Irregular** (un grande + varios pequeños, joint plot, dashboard): `GridSpec`.

**❓ ¿`fig.suptitle` o `ax.set_title` con subplots?**

`suptitle` = título de toda la figura (encima de todos). `ax.set_title` = título por subplot. Combinables.

**❓ ¿Cómo controlo espacio entre subplots?**

`subplots(..., gridspec_kw={'hspace': 0.3, 'wspace': 0.3})` (ratios relativos al tamaño del axes). Con `constrained_layout` suele ser automático.

**❓ ¿`figsize` cómo elegir?**

Para artículos: ancho de columna (típicamente 3.5" para 1 col, 7" para ancho página). Para presentaciones: aspect ratio 16:9 → `(12, 6.75)`.

**❓ ¿`axes.flat` o `axes.flatten()`?**

**`flat`** es iterador (no copia). **`flatten()`** devuelve nuevo array. Para iterar, `flat` ahorra memoria.

## 🔗 Referencias

- VanderPlas cap. 4 § 4.6
- [GridSpec tutorial](https://matplotlib.org/stable/users/explain/axes/arranging_axes.html)

➡️ **Siguiente:** [036 — Legends, colorbars, ticks, anotaciones](../036-matplotlib-legends-colorbars-ticks-anotaciones/README.md)